# 00 - Process Single Year CRD Mortality Data

## Motivation

In this project, we are predicting county-level chronic respiratory disease (CRD) mortality rates in the United States.

We use **age-standardized mortality rates** to ensure fair comparisons across counties with different age distributions.

This notebook extracts the CRD mortality data for each individual year (2012-2019), which will later be combined into a single dataset.

In [1]:
import pandas as pd
import os

## Processing Function

The function below extracts CRD mortality data for a single year:
- Filters for `race_name == 'Total'` (aggregate across all races)
- Filters for `age_name == 'Age-standardized'` (age-standardized mortality rate)
- Keeps only county-level data (`fips > 60`)
- Removes unnecessary columns and renames the target variable

In [2]:
def process_crd_mortality_data(year):
    input_file_path = f"../data/raw/IHME_USA_COD_COUNTY_RACE_ETHN_2000_2019_MX_RESP_BOTH/IHME_USA_COD_COUNTY_RACE_ETHN_2000_2019_MX_{year}_RESP_BOTH_Y2023M06D12.CSV"

    df = pd.read_csv(input_file_path)
    print(f"Loaded {len(df)} rows for year {year}")

    df_filtered = df[(df['race_name'] == 'Total') & (df['age_name'] == 'Age-standardized')]
    print(f"After filtering for Total race and Age-standardized: {len(df_filtered)} rows")

    df_clean = df_filtered.dropna()
    print(f"After dropping NaN: {len(df_clean)} rows")

    df_counties = df_clean[df_clean['fips'] > 60]
    print(f"After filtering for county-level (fips > 60): {len(df_counties)} rows")

    columns_to_drop = ['measure_id', 'location_id', 'fips', 'measure_name', 'race_id',
                       'race_name', 'sex_id', 'sex_name', 'age_group_id', 'age_name',
                       'cause_id', 'cause_name', 'metric_id', 'metric_name', 'upper', 'lower']
    df_final = df_counties.drop(columns=columns_to_drop)

    df_final = df_final.rename(columns={'val': 'crd_mortality_rate'})

    output_file_path = f"../data/processed/crd_single_year/crd_mortality_{year}.csv"
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    df_final.to_csv(output_file_path, index=False)

    print(f"Processed CRD mortality data for {year} saved to {output_file_path}")
    print(f"Final dataset shape: {df_final.shape}")
    print("---")

    return df_final

## Test with a Single Year

In [3]:
df_2012 = process_crd_mortality_data(2012)
df_2012.head()

Loaded 404460 rows for year 2012
After filtering for Total race and Age-standardized: 3210 rows
After dropping NaN: 3178 rows
After filtering for county-level (fips > 60): 3127 rows
Processed CRD mortality data for 2012 saved to ../data/processed/crd_single_year/crd_mortality_2012.csv
Final dataset shape: (3127, 3)
---


,location_name,year,crd_mortality_rate
8292,Autauga County (Alabama),2012,0.000997
8298,Baldwin County (Alabama),2012,0.000669
8304,Barbour County (Alabama),2012,0.000785
8310,Bibb County (Alabama),2012,0.001007
8316,Blount County (Alabama),2012,0.001024


In [4]:
print("CRD Mortality Rate Statistics (2012):")
print(df_2012['crd_mortality_rate'].describe())

CRD Mortality Rate Statistics (2012):
count    3127.000000
mean        0.000737
std         0.000183
min         0.000218
25%         0.000606
50%         0.000724
75%         0.000848
max         0.001740
Name: crd_mortality_rate, dtype: float64


## Process All Years (2012-2019)

We process years 2012-2019 to match the time period of our ACS and atmospheric data.

In [5]:
for year in range(2012, 2020):
    process_crd_mortality_data(year)

Loaded 404460 rows for year 2012
After filtering for Total race and Age-standardized: 3210 rows
After dropping NaN: 3178 rows
After filtering for county-level (fips > 60): 3127 rows
Processed CRD mortality data for 2012 saved to ../data/processed/crd_single_year/crd_mortality_2012.csv
Final dataset shape: (3127, 3)
---
Loaded 404460 rows for year 2013
After filtering for Total race and Age-standardized: 3210 rows
After dropping NaN: 3178 rows
After filtering for county-level (fips > 60): 3127 rows
Processed CRD mortality data for 2013 saved to ../data/processed/crd_single_year/crd_mortality_2013.csv
Final dataset shape: (3127, 3)
---
Loaded 404460 rows for year 2014
After filtering for Total race and Age-standardized: 3210 rows
After dropping NaN: 3178 rows
After filtering for county-level (fips > 60): 3127 rows
Processed CRD mortality data for 2014 saved to ../data/processed/crd_single_year/crd_mortality_2014.csv
Final dataset shape: (3127, 3)
---
Loaded 404460 rows for year 2015
Afte

## Verify Output Files

In [6]:
output_dir = "../data/processed/crd_single_year/"
files = sorted(os.listdir(output_dir))
print("Generated files:")
for f in files:
    if f.endswith('.csv'):
        filepath = os.path.join(output_dir, f)
        df = pd.read_csv(filepath)
        print(f"  {f}: {len(df)} rows")

Generated files:
  crd_mortality_2012.csv: 3127 rows
  crd_mortality_2013.csv: 3127 rows
  crd_mortality_2014.csv: 3127 rows
  crd_mortality_2015.csv: 3127 rows
  crd_mortality_2016.csv: 3127 rows
  crd_mortality_2017.csv: 3127 rows
  crd_mortality_2018.csv: 3127 rows
  crd_mortality_2019.csv: 3127 rows
